In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/vhap")
(ROOT / "decisions").mkdir(parents=True, exist_ok=True)
(ROOT / "data").mkdir(exist_ok=True)
(ROOT / "models" / "checkpoints").mkdir(parents=True, exist_ok=True)

text = """
Project: J26-IT-438 Visual Hierarchy & Attention Predictor
Locked before looking at any test result. Do not change these after evaluation.

1. Baseline (Model 1): TranSalNet_Res, SALICON weights, no training.
   Optional later baseline: OpenCV spectral residual. Not part of Step 1.
2. Main model (Model 2): the same TranSalNet_Res, fine-tuned later on the UEyes webpage subset only.
3. Variant: TranSalNet_Res (ResNet-50). Do not switch to TranSalNet_Dense.
4. Input: pad to 384 wide by 288 tall. Do not stretch the image.
5. Pixel scaling: divide by 255 only. Do not use ImageNet mean/std.
6. Output: the model already applies sigmoid. Values are 0 to 1. Do not sigmoid again.
7. Ground truth saliency: UEyes heatmaps_3s.
   Ground truth fixations: UEyes fixmaps_3s.
   Ignore the train/test flag inside UEyes info.csv. Our own split comes later.
8. Split, used later: by page, seed 42, 70% train / 15% validation / 15% test.
   Test pages are never used for tuning.
9. Element score, used later: mean saliency inside the box.
10. API input, used later: screenshot, plus URL when one exists.
    Playwright is the element source. A detector is only the fallback.
"""
(ROOT / "decisions" / "locked_decisions.txt").write_text(text.strip() + "\n")
print("Saved", ROOT / "decisions" / "locked_decisions.txt")

Saved /content/drive/MyDrive/vhap/decisions/locked_decisions.txt


In [5]:
!nvidia-smi
!df -h /content

Wed Oct 14 16:51:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
%cd /content
!wget -c -O UEyes_dataset.zip "https://zenodo.org/records/8010312/files/UEyes_dataset.zip?download=1"
!md5sum UEyes_dataset.zip

/content
--2026-10-15 14:11:01--  https://zenodo.org/records/8010312/files/UEyes_dataset.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.45.63, 188.185.47.192, 137.138.121.78, ...
Connecting to zenodo.org (zenodo.org)|188.185.45.63|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12912746325 (12G) [application/octet-stream]
Saving to: ‘UEyes_dataset.zip’

UEyes_dataset.zip     0%[                    ]  20.31M   788KB/s    eta 4h 28m ^C
53f585058ea11ba8a08c9f606c89f338  UEyes_dataset.zip


In [2]:
!mkdir -p /content/ueyes_raw
!unzip -q -o UEyes_dataset.zip -d /content/ueyes_raw
!find /content/ueyes_raw -maxdepth 3 -type d
!find /content/ueyes_raw -name info.csv

unzip:  cannot find or open UEyes_dataset.zip, UEyes_dataset.zip.zip or UEyes_dataset.zip.ZIP.
/content/ueyes_raw


In [7]:
!rm -f /content/UEyes_dataset.zip

In [8]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [9]:
!mkdir -p /content/drive/MyDrive/vhap/data
!wget -c -O /content/drive/MyDrive/vhap/data/UEyes_dataset.zip "https://zenodo.org/records/8010312/files/UEyes_dataset.zip?download=1"

--2026-10-15 14:16:20--  https://zenodo.org/records/8010312/files/UEyes_dataset.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.42.113, 188.185.45.63, 188.185.51.60, ...
Connecting to zenodo.org (zenodo.org)|188.185.42.113|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12912746325 (12G) [application/octet-stream]
Saving to: ‘/content/drive/MyDrive/vhap/data/UEyes_dataset.zip’

/content/drive/MyDr 100%[===================>]  12.03G  2.98MB/s    in 59m 50s 

2026-10-15 15:16:10 (3.43 MB/s) - ‘/content/drive/MyDrive/vhap/data/UEyes_dataset.zip’ saved [12912746325/12912746325]



In [10]:
ZIP = "/content/drive/MyDrive/vhap/data/UEyes_dataset.zip"
!ls -lh "$ZIP"
!df -h /content

-rw------- 1 root root 13G Oct 15 15:16 /content/drive/MyDrive/vhap/data/UEyes_dataset.zip
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   33G   76G  31% /


In [17]:
!cp /content/drive/MyDrive/vhap/data/UEyes_dataset.zip /content/UEyes_dataset.zip
!mkdir -p /content/ueyes_raw
!unzip -q -o /content/UEyes_dataset.zip -d /content/ueyes_raw
!find /content/ueyes_raw -maxdepth 3 -type d
!find /content/ueyes_raw -name info.csv

/content/ueyes_raw
/content/ueyes_raw/UEyes_dataset
/content/ueyes_raw/UEyes_dataset/eyetracker_logs
/content/ueyes_raw/UEyes_dataset/scanpaths
/content/ueyes_raw/UEyes_dataset/scanpaths/paths_7s
/content/ueyes_raw/UEyes_dataset/scanpaths/paths_1s
/content/ueyes_raw/UEyes_dataset/scanpaths/paths_3s
/content/ueyes_raw/UEyes_dataset/images
/content/ueyes_raw/UEyes_dataset/saliency_maps
/content/ueyes_raw/UEyes_dataset/saliency_maps/heatmaps_1s
/content/ueyes_raw/UEyes_dataset/saliency_maps/overlay_heatmaps_7s
/content/ueyes_raw/UEyes_dataset/saliency_maps/fixmaps_7s
/content/ueyes_raw/UEyes_dataset/saliency_maps/heatmaps_7s
/content/ueyes_raw/UEyes_dataset/saliency_maps/overlay_heatmaps_1s
/content/ueyes_raw/UEyes_dataset/saliency_maps/fixmaps_3s
/content/ueyes_raw/UEyes_dataset/saliency_maps/overlay_heatmaps_3s
/content/ueyes_raw/UEyes_dataset/saliency_maps/fixmaps_1s
/content/ueyes_raw/UEyes_dataset/saliency_maps/heatmaps_3s
/content/ueyes_raw/__MACOSX
/content/ueyes_raw/__MACOSX/UEyes

In [20]:
from pathlib import Path
import pandas as pd

raw_root = Path("/content/ueyes_raw/UEyes_dataset")
print("Folders:", sorted(p.name for p in raw_root.iterdir() if p.name != "__MACOSX"))
print("CSV files:", list(raw_root.rglob("*.csv"))[:20])
print("Sample images:", sorted(p.name for p in (raw_root / "images").glob("*"))[:8])

Folders: ['.DS_Store', 'README.md', 'eyetracker_logs', 'image_types.csv', 'images', 'saliency_maps', 'scanpaths']
CSV files: [PosixPath('/content/ueyes_raw/UEyes_dataset/image_types.csv'), PosixPath('/content/ueyes_raw/UEyes_dataset/eyetracker_logs/42_kh033_fixations.csv'), PosixPath('/content/ueyes_raw/UEyes_dataset/eyetracker_logs/34_kh055_fixations.csv'), PosixPath('/content/ueyes_raw/UEyes_dataset/eyetracker_logs/45_kh045_fixations.csv'), PosixPath('/content/ueyes_raw/UEyes_dataset/eyetracker_logs/42_kh021_fixations.csv'), PosixPath('/content/ueyes_raw/UEyes_dataset/eyetracker_logs/22_kh054_fixations.csv'), PosixPath('/content/ueyes_raw/UEyes_dataset/eyetracker_logs/52_kh056_fixations.csv'), PosixPath('/content/ueyes_raw/UEyes_dataset/eyetracker_logs/32_kh022_fixations.csv'), PosixPath('/content/ueyes_raw/UEyes_dataset/eyetracker_logs/13_kh038_fixations.csv'), PosixPath('/content/ueyes_raw/UEyes_dataset/eyetracker_logs/23_kh033_fixations.csv'), PosixPath('/content/ueyes_raw/UEyes_d

In [21]:
import pandas as pd
from pathlib import Path

df = pd.read_csv("/content/ueyes_raw/UEyes_dataset/image_types.csv")
print("Columns:", df.columns.tolist())
print(df.head(5).to_string())
for col in df.columns:
    if df[col].dtype == object:
        print(col, "->", df[col].astype(str).unique()[:12])

ParserError: Error tokenizing data. C error: Expected 1 fields in line 23, saw 2


In [22]:
import pandas as pd

path = "/content/ueyes_raw/UEyes_dataset/image_types.csv"
with open(path, errors="replace") as f:
    raw = f.read().splitlines()

print("lines", len(raw))
for i, line in enumerate(raw[:30]):
    print(i, repr(line))

for sep in ["\t", ",", ";", "|", r"\s+"]:
    try:
        trial = pd.read_csv(path, sep=sep, engine="python")
        print("SEP", repr(sep), "shape", trial.shape, "cols", trial.columns.tolist())
    except Exception as e:
        print("SEP", repr(sep), "FAILED", type(e).__name__)

lines 1981
0 'Image Name;Category;Block;Train/Test'
1 '2021b7.png;desktop;0;Train'
2 '460bf9.png;desktop;0;Train'
3 'd99b18.png;desktop;0;Train'
4 '7613f2.png;desktop;0;Train'
5 'a2da71.png;desktop;0;Train'
6 '95b2d2.png;desktop;0;Train'
7 '8b0b52.png;desktop;0;Train'
8 'fbd704.png;desktop;0;Train'
9 '2e93e6.png;desktop;0;Train'
10 'c27e4e.jpg;mobile;0;Train'
11 '536e7f.jpg;mobile;0;Train'
12 '4adea3.jpg;mobile;0;Train'
13 '13af7a.jpg;mobile;0;Train'
14 'e3d203.jpg;mobile;0;Train'
15 '17a8d1.jpg;mobile;0;Train'
16 '58a60d.jpg;mobile;0;Train'
17 '207fd4.jpg;mobile;0;Train'
18 'f639bf.jpg;mobile;0;Train'
19 '708a08.png;poster;0;Train'
20 'b2d2fa.jpg;poster;0;Train'
21 'cced5f.jpg;poster;0;Train'
22 '05e980.jpg;poster;0,00E+00;Train'
23 '9afe5f.png;poster;0;Train'
24 '65576e.jpg;poster;0;Train'
25 '085001.jpg;poster;0;Train'
26 'e7e220.png;poster;0;Train'
27 '6eb097.png;poster;0;Train'
28 '9efc81.png;web;0;Train'
29 '166bc9.png;web;0;Train'
SEP '\t' shape (1980, 1) cols ['Image Name;Categ

In [23]:
import pandas as pd

df = pd.read_csv("/content/ueyes_raw/UEyes_dataset/image_types.csv", sep=";")
web = df[df["Category"].astype(str).str.strip().str.lower().eq("web")].copy()
print("Webpage rows:", len(web))
print(web["Image Name"].head(10).tolist())
print(web["Category"].unique())

Webpage rows: 495
['9efc81.png', '166bc9.png', '72062c.png', '69a8c4.png', '29227b.png', '5c88b8.png', '27d67e.png', '4e81ec.png', '781b7e.png', '1ee232.png']
['web']


In [24]:
import shutil
from pathlib import Path

raw_root = Path("/content/ueyes_raw/UEyes_dataset")
img_dir = raw_root / "images"
sal_dir = raw_root / "saliency_maps" / "heatmaps_3s"
fix_dir = raw_root / "saliency_maps" / "fixmaps_3s"

web_root = Path("/content/drive/MyDrive/vhap/data/ueyes_web")
for sub in ("images", "saliency_maps", "fixations"):
    (web_root / sub).mkdir(parents=True, exist_ok=True)

web.to_csv(web_root / "image_types_web.csv", index=False)

def find_file(folder, stem):
    hits = [p for p in folder.glob(stem + ".*") if not p.name.startswith(".")]
    return hits[0] if hits else None

missing = []
for name in web["Image Name"].astype(str):
    stem = Path(name).stem
    img = find_file(img_dir, stem)
    sal = find_file(sal_dir, stem)
    fix = find_file(fix_dir, stem)
    if not img or not sal or not fix:
        missing.append((stem, bool(img), bool(sal), bool(fix)))
        continue
    shutil.copy2(img, web_root / "images" / img.name)
    shutil.copy2(sal, web_root / "saliency_maps" / (stem + sal.suffix))
    shutil.copy2(fix, web_root / "fixations" / (stem + fix.suffix))

print("Copied images:", len(list((web_root / "images").glob("*"))))
print("Copied saliency maps:", len(list((web_root / "saliency_maps").glob("*"))))
print("Copied fixations:", len(list((web_root / "fixations").glob("*"))))
print("Missing:", len(missing))
print(missing[:10])

Copied images: 495
Copied saliency maps: 495
Copied fixations: 495
Missing: 0
[]


In [25]:
!rm -f /content/UEyes_dataset.zip
!rm -f /content/drive/MyDrive/vhap/data/UEyes_dataset.zip

In [26]:
%cd /content
!git clone https://github.com/LJOVO/TranSalNet.git
%cd /content/TranSalNet
!mkdir -p pretrained_models
!pip -q install gdown
!gdown 14czAAQQcRLGeiddPOM6AaTJTieu6QiHy -O pretrained_models/TranSalNet_Res.pth
!wget -O pretrained_models/resnet50-0676ba61.pth https://download.pytorch.org/models/resnet50-0676ba61.pth
!ls -lh pretrained_models

/content
Cloning into 'TranSalNet'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 58 (delta 6), reused 3 (delta 3), pack-reused 43 (from 1)
Receiving objects: 100% (58/58), 3.14 MiB | 8.89 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/TranSalNet
Downloading...
From (original): https://drive.google.com/uc?id=14czAAQQcRLGeiddPOM6AaTJTieu6QiHy
From (redirected): https://drive.google.com/uc?id=14czAAQQcRLGeiddPOM6AaTJTieu6QiHy&confirm=t&uuid=499404fd-5609-451e-adbf-3e9e096dfc63
To: /content/TranSalNet/pretrained_models/TranSalNet_Res.pth
100% 290M/290M [00:03<00:00, 93.8MB/s]
--2026-10-15 17:14:16--  https://download.pytorch.org/models/resnet50-0676ba61.pth
Resolving download.pytorch.org (download.pytorch.org)... 18.160.143.101, 18.160.143.107, 18.160.143.48, ...
Connecting to download.pytorch.org (download.pytorch.org)|18.160.143.101|:443... connected.
HTTP request sent, awa

In [27]:
import shutil
from pathlib import Path

dst = Path("/content/drive/MyDrive/vhap/models/transalnet_pretrained")
dst.mkdir(parents=True, exist_ok=True)
shutil.copy2("pretrained_models/TranSalNet_Res.pth", dst / "TranSalNet_Res.pth")
shutil.copy2("pretrained_models/resnet50-0676ba61.pth", dst / "resnet50-0676ba61.pth")
print(list(dst.iterdir()))

[PosixPath('/content/drive/MyDrive/vhap/models/transalnet_pretrained/TranSalNet_Res.pth'), PosixPath('/content/drive/MyDrive/vhap/models/transalnet_pretrained/resnet50-0676ba61.pth')]


In [28]:
%cd /content/TranSalNet

import torch
from TranSalNet_Res import TranSalNet

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TranSalNet()
state = torch.load("pretrained_models/TranSalNet_Res.pth", map_location=device, weights_only=False)
model.load_state_dict(state)
model = model.to(device).eval()

x = torch.rand(1, 3, 288, 384, device=device)
with torch.no_grad():
    y = model(x)
print(device, tuple(y.shape), float(y.min()), float(y.max()))

/content/TranSalNet
cpu (1, 1, 288, 384) 0.0006364101427607238 0.5045846104621887


In [29]:
from pathlib import Path

web = Path("/content/drive/MyDrive/vhap/data/ueyes_web")
for sub in ("images", "saliency_maps", "fixations"):
    print(sub, len(list((web / sub).glob("*"))))

weights = Path("/content/drive/MyDrive/vhap/models/transalnet_pretrained")
for name in ("TranSalNet_Res.pth", "resnet50-0676ba61.pth"):
    file = weights / name
    print(name, file.exists(), file.stat().st_size if file.exists() else 0)

images 495
saliency_maps 495
fixations 495
TranSalNet_Res.pth True 290451767
resnet50-0676ba61.pth True 102530333
